In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
# from sklearn.feature_extraction.text import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from sklearn.metrics import accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import joblib
# import re as regex
import string

In [21]:
def remove_stopwords_and_specials(text):
    stop_words = set(stopwords.words('english'))
    words_tokens = word_tokenize(text)
    
    # creates regex of special characters and punctuation
    # special_chars = regex.compile('[@_!#$%^&*()<>?/\|}{~:]')
    punc = string.punctuation

    filtered_sentence = [w for w in words_tokens if not w in stop_words and w not in punc]

    return ' '.join(filtered_sentence)

In [3]:
def clean_data(text):
    # removes stop words
    no_stops = remove_stopwords_and_specials(text.lower())
    


In [4]:
def create_sentiment_integer(sentiment):
    if sentiment == 'negative':
        return -1
    elif sentiment == 'positive':
        return 1
    return 0

In [5]:
df = pd.read_csv('../data/Tweets.csv')
df.head()

,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [6]:
# creates sentiment integer values for each line of text
df['value'] = df['sentiment'].apply(create_sentiment_integer)
df

,textID,text,selected_text,sentiment,value
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,0
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,-1
2,088c60f138,my boss is bullying me...,bullying me,negative,-1
3,9642c003ef,what interview! leave me alone,leave me alone,negative,-1
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,-1
...,...,...,...,...,...
27476,4eac33d1c0,wish we could come see u on Denver husband l...,d lost,negative,-1
27477,4f4c4fc327,I`ve wondered about rake to. The client has ...,", don`t force",negative,-1
27478,f67aae2310,Yay good for both of you. Enjoy the break - y...,Yay good for both of you.,positive,1
27479,ed167662a5,But it was worth it ****.,But it was worth it ****.,positive,1


In [7]:
df['text'][0]

' I`d have responded, if I were going'

In [8]:
# cleans data of stop words, special characters, and punctuation
df['text'] = df['text'].apply(str).apply(remove_stopwords_and_specials)
df['selected_text'] = df['selected_text'].apply(str).apply(remove_stopwords_and_specials)
df

,textID,text,selected_text,sentiment,value
0,cb774db0d1,I responded I going,I responded I going,neutral,0
1,549e992a42,Sooo SAD I miss San Diego,Sooo SAD,negative,-1
2,088c60f138,boss bullying ...,bullying,negative,-1
3,9642c003ef,interview leave alone,leave alone,negative,-1
4,358bd9e861,Sons put releases already bought,Sons,negative,-1
...,...,...,...,...,...
27476,4eac33d1c0,wish could come see u Denver husband lost job ...,lost,negative,-1
27477,4f4c4fc327,I wondered rake The client made clear .NET for...,force,negative,-1
27478,f67aae2310,Yay good Enjoy break probably need hectic week...,Yay good,positive,1
27479,ed167662a5,But worth,But worth,positive,1


In [9]:
tfidf = TfidfVectorizer(strip_accents=None, lowercase=False, preprocessor=None)
X = tfidf.fit_transform(df['text'])

In [10]:
y = df['value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
params_per_model =[
    {
        'LogisticRegression': {
            'C': uniform(0.1, 500),
            'solver': ['lbfgs', 'liblinear', 'saga', 'sag'],
            'penalty': [None, 'l1', 'l2']
        }
    },
    {
        'SupportVectorClass': {
            'C': uniform(0.1, 10),
            'kernel': ['linear', 'poly', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    },
]

In [11]:
# {'C': 1.1482306800048692, 'penalty': 'l2', 'solver': 'sag'}
lgr = LogisticRegression()
lgr.set_params(C=1.1482306800048692, penalty='l2', solver='sag')

LogisticRegression(C=1.1482306800048692, solver='sag')

In [12]:
# {'C': 0.9779819100489128, 'gamma': 'auto', 'kernel': 'linear'}
svc = SVC()
svc.set_params(C=0.9779819100489128, gamma='auto', kernel='linear')


SVC(C=0.9779819100489128, gamma='auto', kernel='linear')

In [13]:
lgr.fit(X_train, y_train)

LogisticRegression(C=1.1482306800048692, solver='sag')

In [14]:
svc.fit(X_train, y_train)

SVC(C=0.9779819100489128, gamma='auto', kernel='linear')

In [15]:
# predictions
preds_log = lgr.predict(X_test)
preds_svc = svc.predict(X_test)

In [16]:
# accuracy scores
print(f"Logistic Regression Accuracy Score: {accuracy_score(preds_log, y_test)}")
print(f"Support Vector Classification Accuracy Score: {accuracy_score(preds_svc, y_test)}")

Logistic Regression Accuracy Score: 0.6677986658580958
Support Vector Classification Accuracy Score: 0.6789569436021832


In [17]:
joblib.dump(lgr, '../models/logistic_regression_model.pkl')
joblib.dump(svc, '../models/support_vector_classification_model.pkl')

['../models/support_vector_classification_model.pkl']

In [18]:
saved_sentiment_csv = df.drop(['textID', 'selected_text'], axis=1)
saved_sentiment_csv

,text,sentiment,value
0,I responded I going,neutral,0
1,Sooo SAD I miss San Diego,negative,-1
2,boss bullying ...,negative,-1
3,interview leave alone,negative,-1
4,Sons put releases already bought,negative,-1
...,...,...,...
27476,wish could come see u Denver husband lost job ...,negative,-1
27477,I wondered rake The client made clear .NET for...,negative,-1
27478,Yay good Enjoy break probably need hectic week...,positive,1
27479,But worth,positive,1


In [19]:
saved_sentiment_csv.to_csv('../data/Cleaned_Tweets_Dataset.csv')

In [20]:
f = pd.read_csv('../data/Cleaned_Tweets_Dataset.csv')
f

,Unnamed: 0,text,sentiment,value
0,0,I responded I going,neutral,0
1,1,Sooo SAD I miss San Diego,negative,-1
2,2,boss bullying ...,negative,-1
3,3,interview leave alone,negative,-1
4,4,Sons put releases already bought,negative,-1
...,...,...,...,...
27476,27476,wish could come see u Denver husband lost job ...,negative,-1
27477,27477,I wondered rake The client made clear .NET for...,negative,-1
27478,27478,Yay good Enjoy break probably need hectic week...,positive,1
27479,27479,But worth,positive,1
